# 02 — Retrieval Evaluation

Evaluates minsearch, FAISS, and RRF across:
- 2 ground truth files: `gpt-5.4-mini`, `gpt-5.6-luna`
- 2 embedding models: `all-MiniLM-L6-v2`, `multi-qa-MiniLM-L6-cos-v1`

Saves all results to `data/retrieval-eval-results.csv`.

In [6]:
import sys, os, importlib
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv(ROOT / '.envrc')

True

In [7]:
def hit_rate_mrr(search_fn, ground_truth, num_results=10):
    hits = 0
    mrr_sum = 0.0
    for _, row in tqdm(ground_truth.iterrows(), total=len(ground_truth), leave=False):
        results = search_fn(row['question'], num_results=num_results)
        result_ids = [str(r['id']) for r in results]
        expected = str(row['movie_id'])
        if expected in result_ids:
            hits += 1
            rank = result_ids.index(expected) + 1
            mrr_sum += 1 / rank
    n = len(ground_truth)
    return round(hits / n, 4), round(mrr_sum / n, 4)

In [8]:
GT_MODELS = ['gpt-5.4-mini', 'gpt-5.6-luna']
EMBEDDING_MODELS = [
    'sentence-transformers/all-MiniLM-L6-v2',
    'sentence-transformers/multi-qa-MiniLM-L6-cos-v1',
]

all_rows = []

for gt_model in GT_MODELS:
    gt = pd.read_csv(ROOT / f'data/ground-truth-retrieval-{gt_model}.csv')
    print(f'\n=== Ground truth: {gt_model} ({len(gt)} pairs) ===')

    for emb_model in EMBEDDING_MODELS:
        print(f'  Loading indexes with {emb_model} ...')
        os.environ['EMBEDDING_MODEL'] = emb_model

        # Reload ingest to rebuild FAISS with the current embedding model
        import movie_assistant.ingest as ingest
        importlib.reload(ingest)

        emb_short = emb_model.split('/')[-1]

        print(f'  minsearch ...')
        hr, mrr = hit_rate_mrr(ingest.search_minsearch, gt)
        all_rows.append({'gt_model': gt_model, 'embedding': emb_short, 'method': 'minsearch', 'hit_rate': hr, 'mrr': mrr})
        print(f'    hit_rate={hr}, mrr={mrr}')

        print(f'  FAISS ({emb_short}) ...')
        hr, mrr = hit_rate_mrr(ingest.search_faiss, gt)
        all_rows.append({'gt_model': gt_model, 'embedding': emb_short, 'method': 'faiss', 'hit_rate': hr, 'mrr': mrr})
        print(f'    hit_rate={hr}, mrr={mrr}')

        print(f'  RRF ...')
        hr, mrr = hit_rate_mrr(ingest.search_rrf, gt)
        all_rows.append({'gt_model': gt_model, 'embedding': emb_short, 'method': 'rrf', 'hit_rate': hr, 'mrr': mrr})
        print(f'    hit_rate={hr}, mrr={mrr}')

results_df = pd.DataFrame(all_rows)
out = ROOT / 'data/retrieval-eval-results.csv'
results_df.to_csv(out, index=False)
print(f'\nSaved to {out}')


=== Ground truth: gpt-5.4-mini (6000 pairs) ===
  Loading indexes with sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

  minsearch ...


  0%|          | 0/6000 [00:00<?, ?it/s]

...

    hit_rate=0.3875, mrr=0.1931
  FAISS (multi-qa-MiniLM-L6-cos-v1) ...


  0%|          | 0/6000 [00:00<?, ?it/s]

    hit_rate=0.5557, mrr=0.3569
  RRF ...


  0%|          | 0/6000 [00:00<?, ?it/s]

    hit_rate=0.5815, mrr=0.3605

Saved to /Users/I556249/PycharmProjects/llm-capstone/data/retrieval-eval-results.csv


In [9]:
# Results table sorted by MRR
results_df.sort_values('mrr', ascending=False).reset_index(drop=True)

,gt_model,embedding,method,hit_rate,mrr
0,gpt-5.6-luna,all-MiniLM-L6-v2,faiss,0.5973,0.3856
1,gpt-5.6-luna,all-MiniLM-L6-v2,rrf,0.6092,0.3810
2,gpt-5.6-luna,multi-qa-MiniLM-L6-cos-v1,rrf,0.5815,0.3605
3,gpt-5.6-luna,multi-qa-MiniLM-L6-cos-v1,faiss,0.5557,0.3569
4,gpt-5.4-mini,all-MiniLM-L6-v2,faiss,0.5407,0.3379
5,gpt-5.4-mini,all-MiniLM-L6-v2,rrf,0.5533,0.3326
6,gpt-5.4-mini,multi-qa-MiniLM-L6-cos-v1,faiss,0.5055,0.3149
7,gpt-5.4-mini,multi-qa-MiniLM-L6-cos-v1,rrf,0.5320,0.3114
8,gpt-5.6-luna,all-MiniLM-L6-v2,minsearch,0.3875,0.1931
9,gpt-5.6-luna,multi-qa-MiniLM-L6-cos-v1,minsearch,0.3875,0.1931
